In [0]:
from delta.tables import DeltaTable

staging_table_name = 'incremental_load_catalog.default.staging_table'
target_table_name = 'incremental_load_catalog.default.target_table'


stage_df = spark.read.table(staging_table_name)

In [0]:
if not spark._jsparkSession.catalog().tableExists(target_table_name):
    stage_df.write.format('delta').mode('overwrite').saveAsTable(target_table_name)
else:
    target_table = DeltaTable.forName(spark, target_table_name)
    condition = 'stage.tracking_num = target.tracking_num'

    target_table.alias('target') \
        .merge(stage_df.alias('stage'), condition) \
        .whenMatchedDelete() \
        .execute()
    
    stage_df.write.format('delta').mode('append').saveAsTable(target_table_name)

